# Ocular adversarial XAI: cross-session retraining of Xception + ArcFace (2 seeds)

One Kaggle session that retrains the baseline model under the **cross-session protocol** (train and validation on enrollment-phase images, test on every verification-phase image) with the training recipe of `configs/cross_session.yml`, for two seeds at once (seed 1 on GPU 0, seed 2 on GPU 1), and evaluates both models: test accuracy (04) and verification P2 + P3 as EER and TAR at fixed FAR (15). It trains from ImageNet weights: no checkpoint input is needed.

**Before you start**
1. The training-recipe PR (03 `--seed`, `--resume`, `--reshuffle`, `--save-weights`, the run record `run_<tag>.json`, `configs/cross_session.yml`, 04 `--class-index`) must be merged into `BRANCH` (cell 1, default `main`). Cell 6 stops if a script, flag or the config is missing.
2. Settings: accelerator **GPU T4 x2**, Internet **on** (to clone the repo), secret `github_token` attached.
3. Input: `mubashirulhassan00/visob-augmented-recent` only. To continue a run that Kaggle stopped, also attach the earlier version's output and set `PREVIOUS_OUTPUT` (see *Resume* below).
4. Keep this notebook **private**. Never upload `results.zip` or the checkpoints to a public place: they are derived from VISOB.

**Workflow**
- First run all cells interactively with `RUN_FULL = False` (about 20-40 min): setup, checks, the manifest and the cross-session split with its checks, smoke trainings of both seeds at the same time on a 20-class tree, a test of `--resume` (a training is stopped after its first epoch and continued), smoke runs of 04 and 15, and the time estimate of the full run.
- Then set `RUN_FULL = True` (cell 2) and run with **Save Version > Save & Run All**, so the outputs are kept. The checks and smoke runs are repeated, and the notebook refuses to start the full runs if the estimate for 30 epochs (the cap) of both seeds does not fit before `HARD_STOP_H`.
- Download `results.zip` from the Output panel; the checkpoints are left out of it (cell 14 lists them). Never import TensorFlow in a notebook cell: every TensorFlow script runs in its own process.

**Expected times.** One epoch on the random split took about 17 min on a T4 (25 and 27 Oct; 10 epochs in 2.9 h). The cross-session training set is smaller (only enrollment images, a quarter of their buckets held out for validation), so an epoch should be shorter; the smoke run measures it with both trainings running together (they share the 4 CPU cores that load and augment the images) and prints the estimate. Early stopping (patience 5) may end a run before the 30-epoch cap, but the 12-hour check assumes all 30.

| Cell | GPU 0 | GPU 1 | Time |
|---|---|---|---|
| 7. Data | | | ~20 min (01 manifest, 02 cross-session split, open-set list, 20-class smoke tree) |
| 9. Smoke A | 03 smoke, seed 1 (2 epochs, 20 classes) | 03 smoke, seed 2 | ~10 min |
| 10. Smoke B | `--resume` test: stopped after epoch 1, then continued | 04 and 15 smoke on the seed-1 smoke model | ~10 min |
| 12. Full (`RUN_FULL`) | 03 seed 1 (cap 30 epochs), then 04 and 15 P2 + P3 | the same for seed 2 | from the smoke estimate |

**Resume.** Kaggle stops a session after 12 hours. Training writes a backup after every epoch (`trained_models/backup_cs_s<seed>`), and the job runner stops every job at `HARD_STOP_H` so the version can still save its outputs. To continue: open the notebook, *Add Input* > *Your Work* > this notebook's stopped version (its output), set `PREVIOUS_OUTPUT` (cell 2) to the folder that appears under `/kaggle/input`, and run again with Save & Run All. Cell 8 copies the earlier models, backups and results, checks that the split is identical, and each seed continues from its last finished epoch (finished steps are skipped).

## 1-3. Setup, settings and the commands of the session

In [ ]:
# 1. Setup: clone the repo at BRANCH, install the pinned tf-keras, record the commit
import os, subprocess, time
SESSION_START = time.time()   # Kaggle ends a session after 12 h; the job runner (cell 4) watches this clock

BRANCH = 'main'   # must hold the training-recipe PR (03 --seed/--resume/--reshuffle/--save-weights, the run record,
                  # configs/cross_session.yml, 04 --class-index); cell 6 stops if something is missing

!pip install -q --no-deps tf-keras==2.20.1

from kaggle_secrets import UserSecretsClient
tok = UserSecretsClient().get_secret('github_token')
REPO = '/tmp/ocular-adv-xai'
url = f'https://{tok}@github.com/nandininautiyal/ocular-adv-xai.git'
if not os.path.exists(REPO):
    !git clone -q -b {BRANCH} {url} {REPO}
    !git -C {REPO} remote set-url origin https://github.com/nandininautiyal/ocular-adv-xai.git
else:
    # a reused session may have another branch checked out: always land exactly on BRANCH
    !git -C {REPO} fetch -q {url} {BRANCH}
    !git -C {REPO} checkout -q -f -B {BRANCH} FETCH_HEAD
%cd {REPO}
!git log -3 --oneline
COMMIT = subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()
print('branch:', BRANCH, '| commit:', COMMIT)
!nvidia-smi -L

In [ ]:
# 2. Settings: the only cell to edit
RUN_FULL = False      # False: checks, data, smoke runs and the time estimate. True: also the full runs
                      # (use Save Version > Save & Run All)
SEEDS = [1]           # one model per seed, trained at the same time: the i-th seed on GPU i; the minimum
                      # for the BTP is seed 1 (add 2 later)
MAX_FRAME = 1009      # 02 and 15 --max-frame: leave out the frames numbered 1010 and up (16 found the
                      # enrollment ones synthetic); None keeps every frame
CFG = 'configs/cross_session.yml'   # the recipe: cap 30 epochs, early stopping (patience 5, best weights restored),
                                    # learning rate x0.2 after 2 epochs without improvement, down to 1e-6
TAG = 'cs'            # the full models are tagged cs_s1, cs_s2: Xception_ArcFace_cs_s1.keras, run_cs_s1.json, ...
SMOKE = 'smoke'       # tag prefix of the smoke runs (smoke_s1, smoke_s2, smoke_resume)
SMOKE_CLASSES = 20    # classes of the smoke tree (02 --max-classes), so that a smoke epoch takes a few minutes
SMOKE_EPOCHS = 2      # epochs of each smoke training; the last one times a steady epoch (the first includes warm-up)
RESUME_TEST_EPOCHS = 3     # the --resume test: stopped after its first epoch, then run again to finish all 3
N_BOOT = 1000         # bootstrap resamples of 15 (the smoke run uses 100)
PREVIOUS_OUTPUT = None     # resume: the folder under /kaggle/input that holds an earlier version's output
                           # (it has trained_models/ and splits/), e.g. '/kaggle/input/<this-notebook>'
START_IF_TOO_LONG = False  # True: start the full runs even if the estimate ends after HARD_STOP_H; they are stopped
                           # there with a backup of the last finished epoch, and the next session resumes them
HARD_STOP_H = 11.25   # Kaggle stops a session after 12 h; any job still running at this point is stopped, so that
                      # the version can still save its outputs
ZIP_MAX_MB = 100      # files larger than this stay out of results.zip (checkpoints always do)

W = '/kaggle/working'
LOGS = f'{W}/logs'
N_CLASSES = 300                                         # VISOB IDs 1141-3279 plus 7000
MODELS, SMOKE_MODELS = f'{W}/trained_models', f'{W}/smoke_models'
CS_TREE, CS_MANIFEST = '/tmp/ocular_cross_session', f'{W}/manifest_split_cross_session.csv'   # 02's defaults
SMALL_TREE, SMALL_MANIFEST = '/tmp/ocular_cs_small', f'{W}/manifest_split_cs_small.csv'
SPLITS = f'{W}/splits'
OPEN_SET = f'{SPLITS}/open_set_test.csv'
EVAL, VERIF = f'{W}/eval_clean', f'{W}/verification'

In [ ]:
# 3. Every command of the session, in one place (nothing runs in this cell)
MF = f' --max-frame {MAX_FRAME}' if MAX_FRAME is not None else ''   # 02 (cross_session, open_set) and 15
def tag(seed, prefix=TAG):
    return f'{prefix}_s{seed}'

def ckpt(models, t):
    return f'{models}/Xception_ArcFace_{t}.keras'

train03 = f'python -u scripts/03_train_baseline.py --cfg {CFG} --resume --reshuffle --save-weights --verbose 2'
small03 = f'{train03} --split-dir {SMALL_TREE} --n-classes {SMOKE_CLASSES} --out-dir {SMOKE_MODELS}'
eval04 = 'python -u scripts/04_eval_clean.py'
verif15 = f'python -u scripts/15_eval_verification.py{MF} --modes p2,p3 --manifest {W}/manifest_raw.csv --p3-list {OPEN_SET}'
s1 = tag(SEEDS[0], SMOKE)       # 04 and 15 are smoke-tested on the first seed's smoke model
CMD = {
    # data, in the foreground (cell 7)
    '01_manifest':      f'python -u scripts/01_build_manifest.py --out {W}/manifest_raw.csv',
    '02_cross_session': f'python -u scripts/02_make_splits.py --protocol cross_session{MF} --write-split-csvs {SPLITS}',
    '02_open_set':      f'python -u scripts/02_make_splits.py --protocol open_set{MF} --write-split-csvs {SPLITS}',
    '02_small':         f'python -u scripts/02_make_splits.py --protocol cross_session{MF} --max-classes {SMOKE_CLASSES} '
                        f'--out-dir {SMALL_TREE} --out-manifest {SMALL_MANIFEST}',
}
# smoke A (cell 9): every seed's training for SMOKE_EPOCHS epochs on the small tree, all at once
for s in SEEDS:
    CMD[f'03_{tag(s, SMOKE)}'] = f'{small03} --epochs {SMOKE_EPOCHS} --seed {s} --tag {tag(s, SMOKE)}'
# smoke B (cell 10): the same command twice; the first run is stopped after its first epoch
resume03 = f'{small03} --epochs {RESUME_TEST_EPOCHS} --seed {SEEDS[0]} --tag {SMOKE}_resume'
CMD['03_resume_a'] = CMD['03_resume_b'] = resume03
CMD['04_smoke'] = (f'{eval04} --split-dir {SMALL_TREE} --n-classes {SMOKE_CLASSES} --ckpt {ckpt(SMOKE_MODELS, s1)} '
                   f'--class-index {SMOKE_MODELS}/class_index_{s1}.json --manifest {SMALL_MANIFEST} --out-dir {EVAL} '
                   f'--tag {s1}')
CMD['15_smoke'] = (f'{verif15} --ckpt {ckpt(SMOKE_MODELS, s1)} --class-index {SMOKE_MODELS}/class_index_{s1}.json '
                   f'--max-subjects 10 --n-boot 100 --out-dir {VERIF} --tag {s1}')
# full (cell 12), one chain per seed and GPU: train, then the test accuracy and verification P2 + P3
for s in SEEDS:
    t = tag(s)
    CMD[f'03_{t}'] = f'{train03} --split-dir {CS_TREE} --n-classes {N_CLASSES} --seed {s} --out-dir {MODELS} --tag {t}'
    CMD[f'04_{t}'] = (f'{eval04} --split-dir {CS_TREE} --n-classes {N_CLASSES} --ckpt {ckpt(MODELS, t)} '
                      f'--class-index {MODELS}/class_index_{t}.json --manifest {CS_MANIFEST} --out-dir {EVAL} --tag {t}')
    CMD[f'15_{t}'] = (f'{verif15} --ckpt {ckpt(MODELS, t)} --class-index {MODELS}/class_index_{t}.json '
                      f'--n-boot {N_BOOT} --out-dir {VERIF} --tag {t}')
# a background step runs only if these steps succeeded (in its own cell or an earlier one)
NEEDS = {'03_resume_a': [f'03_{s1}'], '03_resume_b': [f'03_{s1}'], '04_smoke': [f'03_{s1}'], '15_smoke': [f'03_{s1}'],
         **{f'{step}_{tag(s)}': [f'03_{tag(s)}'] for s in SEEDS for step in ('04', '15')}}
for name, cmd in CMD.items():
    print(f'{name:17} {cmd}')

## 4-5. Helpers and checks

The job runner starts one chain of steps per GPU in the background. Each step writes `logs/<step>.log`, `logs/<step>.start` and `logs/<step>.exit`, runs only if the steps it needs succeeded, and is stopped at `HARD_STOP_H` so the version can still save its outputs (after waiting, at most 4 min, until no checkpoint is being written: a checkpoint cut in half could not be resumed). The check functions are used after the data steps (cell 7), after the smoke runs (cell 11) and at the end (cell 15).

In [ ]:
# 4. Helpers: foreground commands, background jobs (one GPU each), the 12-hour guard, step and epoch timings
import glob, hashlib, json, re, shutil, signal
import numpy as np
import pandas as pd
os.makedirs(LOGS, exist_ok=True)
FG = {}      # exit status of the foreground steps run with sh(..., check=False)

def hours():
    return (time.time() - SESSION_START) / 3600

def sh(cmd, check=True):
    # run cmd from the repo with its output shown here; check: stop the notebook if it fails
    print('$', cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=REPO, executable='/bin/bash', text=True,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in p.stdout:
        print(line, end='', flush=True)
    code = p.wait()
    assert code == 0 or not check, f'exit code {code}: {cmd}'
    return code

def py(snippet):
    # (exit code, last output line) of a Python snippet run in a fresh process (keeps TensorFlow out of the notebook)
    r = subprocess.run(['python', '-c', snippet], cwd=REPO, capture_output=True, text=True)
    out = r.stdout.strip().splitlines() or r.stderr.strip().splitlines() or ['']
    return r.returncode, out[-1]

def load(path):
    with open(path) as f:
        return json.load(f)

def tail(path, n=12):
    try:
        with open(path, errors='ignore') as f:
            lines = [line.rstrip() for line in f if line.strip()]
    except FileNotFoundError:
        return ['(no log)']
    return lines[-n:] or ['(empty log)']

def status(name):
    # 0 ok, another number failed, 'skipped' (a step it needs failed), None (not run, or stopped)
    try:
        text = open(f'{LOGS}/{name}.exit').read().strip()
    except FileNotFoundError:
        return None
    return int(text) if text.lstrip('-').isdigit() else text

def seconds(name):
    # wall-clock seconds of a finished background step (from its .start file to its .exit file), else None
    try:
        start = float(open(f'{LOGS}/{name}.start').read())
        return os.path.getmtime(f'{LOGS}/{name}.exit') - start
    except (FileNotFoundError, ValueError):
        return None

EPOCH_LINE = re.compile(r'^\s*\d+/\d+ - (\d+)s - .*val_loss')
def epoch_seconds(name):
    # seconds of every epoch in a 03 log, validation included (Keras --verbose 2 prints one
    # 'steps/steps - 95s - loss: ... - val_loss: ...' line per epoch)
    try:
        with open(f'{LOGS}/{name}.log', errors='ignore') as f:
            return [int(m.group(1)) for m in map(EPOCH_LINE.match, f) if m]
    except FileNotFoundError:
        return []

def csv_rows(path):
    # data rows of a CSV that may still be written (CSVLogger creates it empty at the start of training)
    try:
        with open(path, errors='ignore') as f:
            return max(sum(1 for line in f if line.strip()) - 1, 0)
    except FileNotFoundError:
        return 0

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(2**24), b''):
            h.update(block)
    return h.hexdigest()

def mtime(path):
    # modification time, 0 for a file that has just been renamed or removed (TensorFlow writes temporary files)
    try:
        return os.path.getmtime(path)
    except OSError:
        return 0

def backup_written(folder):
    # True once tf.keras BackupAndRestore has finished a backup in folder: TensorFlow's CheckpointManager writes
    # the small 'checkpoint' state file after the checkpoint itself
    return bool(glob.glob(f'{folder}/**/checkpoint', recursive=True))

def settle(folders, quiet=60, limit=240, every=5):
    # wait (at most limit s) until no file in folders has been written for quiet s, so that a stop does not cut a
    # checkpoint in half: Keras writes Xception_ArcFace_<tag>.keras in place, and a truncated best checkpoint
    # cannot be resumed or evaluated (backups and the 'checkpoint' state file are replaced atomically)
    t0 = time.time()
    while time.time() - t0 < limit:
        files = [p for d in folders for p in glob.glob(f'{d}/**', recursive=True) if os.path.isfile(p)]
        if time.time() - max([mtime(p) for p in files] or [0]) > quiet:
            return time.time() - t0
        time.sleep(every)
    print(f'files still being written after {limit} s; stopping anyway')
    return time.time() - t0

def launch(label, names, gpu=None):
    # run the steps CMD[name] one after the other in a background job on one GPU (gpu=None: CPU only)
    lines = [f'cd {REPO}', f"export CUDA_VISIBLE_DEVICES={'' if gpu is None else gpu}",
             'export PYTHONUNBUFFERED=1 TF_CPP_MIN_LOG_LEVEL=1']
    for name in names:
        for ext in ('log', 'exit', 'start'):
            if os.path.exists(f'{LOGS}/{name}.{ext}'):
                os.remove(f'{LOGS}/{name}.{ext}')
        needs = NEEDS.get(name, [])
        ok = ' && '.join(f'[ "$(cat {LOGS}/{n}.exit 2>/dev/null)" = 0 ]' for n in needs) or 'true'
        lines.append(f'if {ok}; then date +%s > {LOGS}/{name}.start; {CMD[name]} > {LOGS}/{name}.log 2>&1; '
                     f'echo $? > {LOGS}/{name}.exit; '
                     f'else echo "skipped: needs {" ".join(needs)}" > {LOGS}/{name}.log; '
                     f'echo skipped > {LOGS}/{name}.exit; fi')
    proc = subprocess.Popen(['bash', '-c', '\n'.join(lines)], start_new_session=True)
    print(f'{label}: {" -> ".join(names)}', flush=True)
    return {'label': label, 'names': names, 'proc': proc}

def stop(jobs):
    # stop the running jobs with all their processes (each job is its own process group): SIGTERM, then SIGKILL
    running = [j for j in jobs if j['proc'].poll() is None]
    for sig in (signal.SIGTERM, signal.SIGKILL):
        for job in running:
            try:
                os.killpg(job['proc'].pid, sig)
            except ProcessLookupError:     # every process of that job has ended
                pass
        if sig == signal.SIGTERM:
            time.sleep(30)

def wait(jobs, every=60, also=()):
    # wait for jobs, printing the last log line of each running step every few seconds (also for the jobs in
    # also, which keep running); stop every job at HARD_STOP_H; print and return {step: status}
    while any(j['proc'].poll() is None for j in jobs):
        time.sleep(every)
        if hours() > HARD_STOP_H:
            print(f'{hours():.2f} h: past HARD_STOP_H = {HARD_STOP_H}, stopping every job so the outputs can be saved '
                  f'(first waiting until no checkpoint is being written)', flush=True)
            settle([MODELS, SMOKE_MODELS])
            stop(list(jobs) + list(also))
            break
        parts = []
        for j in list(jobs) + list(also):
            step = next((n for n in j['names'] if status(n) is None), None) if j['proc'].poll() is None else None
            parts.append(f"{j['label']} {step}: {tail(f'{LOGS}/{step}.log', 1)[0][:150]}" if step
                         else f"{j['label']}: done")
        print(time.strftime('%H:%M'), f'({hours():.1f} h)', ' | '.join(parts), flush=True)
    codes = {n: status(n) for j in jobs for n in j['names']}
    for name, code in codes.items():
        word = {0: 'ok', None: 'NOT RUN (or stopped)', 'skipped': 'SKIPPED (a step it needs failed)'}.get(
            code, f'FAILED (exit {code})')
        print(f'\n---- {name}: {word}')
        print('\n'.join(tail(f'{LOGS}/{name}.log', 12 if code == 0 else 40)))
    return codes

In [ ]:
# 5. Checks of each step's outputs; each returns [(check, passed, detail, hard)], hard = must pass
# fields of run_<tag>.json, which 03 writes when training ends (before its test evaluation)
RUN_KEYS = ['finished', 'git_commit', 'git_dirty', 'argv', 'args', 'cfg', 'versions', 'split_dir', 'images_per_split',
            'images_by_phase', 'class_list_sha256', 'epochs_run', 'best_epoch', 'best_val_loss', 'stopped_early',
            'minutes', 'resumed', 'checkpoint', 'checkpoint_sha256', 'weights', 'weights_sha256']

def same_commit(value):
    # the commit a script recorded is this session's (some scripts store a short hash; a dirty mark fails)
    return isinstance(value, str) and len(value) >= 7 and COMMIT.startswith(value)

def split_counts(manifest):
    # images per split and phase, from a 02 manifest
    return pd.read_csv(manifest).groupby(['split', 'phase']).size().unstack(fill_value=0)

def checks_split(manifest, n_classes):
    # 02 --protocol cross_session: test = verification images only, train/validation = enrollment images only,
    # every image in exactly one split, whole (subject, condition, eye) buckets, every class in train
    m = pd.read_csv(manifest)
    name = os.path.basename(manifest)
    phases = {s: set(m.phase[m.split == s]) for s in ('train', 'validation', 'test')}
    raw = pd.read_csv(f'{W}/manifest_raw.csv')
    raw = raw[raw.subject.between(1141, 3279) | (raw.subject == 7000)]
    if n_classes < N_CLASSES:                          # the smoke tree keeps the first n classes
        raw = raw[raw.subject.isin(sorted(raw.subject.unique())[:n_classes])]
    bucket = ['subject', 'condition', 'eye']
    shared = (set(map(tuple, m[m.split == 'train'][bucket].values))
              & set(map(tuple, m[m.split == 'validation'][bucket].values)))
    per_split = m.split.value_counts().to_dict()
    no_test = n_classes - m.subject[m.split == 'test'].nunique()
    return [(f'02 {name}: test = verification images only', phases['test'] == {'verification'},
             str(sorted(phases['test'])), True),
            (f'02 {name}: train and validation = enrollment images only',
             phases['train'] == phases['validation'] == {'enrollment'},
             f"train {sorted(phases['train'])}, validation {sorted(phases['validation'])}", True),
            (f'02 {name}: every image of the {n_classes} classes in exactly one split',
             m.relpath.is_unique and set(m.relpath) == set(raw.relpath) and set(m.split) == {'train', 'validation', 'test'},
             f'{len(m)} rows, {m.relpath.nunique()} relpaths, {len(raw)} manifest images; {per_split}', True),
            (f'02 {name}: {n_classes} classes in train', m.subject[m.split == 'train'].nunique() == n_classes,
             f"{m.subject[m.split == 'train'].nunique()} classes", True),
            (f'02 {name}: no (subject, condition, eye) bucket in both train and validation', not shared,
             f'{len(shared)} shared', True),
            (f'02 {name}: every class has test images', no_test == 0, f'{no_test} classes without test images', False)]

def checks_split_csvs():
    # the lists written by --write-split-csvs: disjoint, and the same images as the manifest
    lists = {s: pd.read_csv(f'{SPLITS}/cross_session_{s}.csv') for s in ('train', 'validation', 'test')}
    m = pd.read_csv(CS_MANIFEST)
    overlap = sum(len(set(lists[a].relpath) & set(lists[b].relpath))
                  for a, b in (('train', 'validation'), ('train', 'test'), ('validation', 'test')))
    same = all(set(lists[s].relpath) == set(m.relpath[m.split == s]) for s in lists)
    return [('02 split lists: no relpath in two splits, same images as the manifest', overlap == 0 and same,
             f'{overlap} relpaths in two lists', True)]

def checks_03(t, models, n_classes, max_epochs, seed):
    files = {k: f'{models}/{stem}_{t}.{ext}' for k, (stem, ext) in {
        'ckpt': ('Xception_ArcFace', 'keras'), 'weights': ('Xception_ArcFace', 'weights.h5'),
        'class_index': ('class_index', 'json'), 'log': ('log', 'csv'), 'history': ('history', 'csv'),
        'run': ('run', 'json')}.items()}
    missing = [os.path.basename(p) for p in files.values() if not os.path.exists(p)]
    out = [(f'03 {t}: checkpoint, weights, class index, logs and run record', not missing, ', '.join(missing), True)]
    if missing:
        return out
    run, idx = load(files['run']), load(files['class_index'])
    res = load(f'{models}/results_{t}.json') if os.path.exists(f'{models}/results_{t}.json') else {}
    log = pd.read_csv(files['log'])
    absent = [k for k in RUN_KEYS if k not in run]
    out.append((f'03 {t}: run record has every field, training finished', not absent and run.get('finished') is True,
                ', '.join(absent) or f"finished {run.get('finished')}", True))
    names = sorted(idx, key=idx.get)
    sha = hashlib.sha256('\n'.join(names).encode('utf-8')).hexdigest()   # 03: sorted names joined with newlines
    out.append((f'03 {t}: class index of {n_classes} classes, indices 0..n-1 in sorted folder order, sha256 recorded',
                len(idx) == n_classes and sorted(idx.values()) == list(range(n_classes)) and names == sorted(names)
                and run.get('class_list_sha256') == sha, f'{len(idx)} classes, class-list sha256 {sha[:12]}', True))
    out.append((f'03 {t}: commit and clean checkout', same_commit(run.get('git_commit')) and run.get('git_dirty') is False,
                f"{run.get('git_commit')}, dirty {run.get('git_dirty')}", True))
    args = run.get('args') or {}
    out.append((f'03 {t}: seed {seed}, --resume, --reshuffle and --save-weights recorded',
                args.get('seed') == seed and all(args.get(k) is True for k in ('resume', 'reshuffle', 'save_weights')),
                str({k: args.get(k) for k in ('seed', 'resume', 'reshuffle', 'save_weights', 'cfg')}), True))
    versions = run.get('versions') or {}
    out.append((f'03 {t}: TensorFlow 2.20 with tf-keras 2.20.1',
                str(versions.get('tensorflow', '')).startswith('2.20.') and versions.get('tf_keras') == '2.20.1',
                str(versions), True))
    epochs = log.epoch.tolist() if 'epoch' in log else []
    out.append((f'03 {t}: at most {max_epochs} epochs, each logged once (a resumed run appends)',
                0 < len(epochs) <= max_epochs and epochs == list(range(len(epochs))),
                f'log epochs {epochs[:3]}...{epochs[-2:]}', True))
    out.append((f'03 {t}: epochs_run of the run record = epochs in the log', run.get('epochs_run') == len(epochs),
                f"epochs_run {run.get('epochs_run')}, {len(epochs)} logged", False))
    best_row = log.val_loss.idxmin() if 'val_loss' in log and len(log) else None
    best_ok = best_row is not None and np.isclose(run.get('best_val_loss') or np.nan, log.val_loss.min(), rtol=1e-5) \
        and run.get('best_epoch') == int(log.epoch[best_row]) + 1
    out.append((f'03 {t}: best epoch (1-based) and best val_loss match the log', bool(best_ok),
                f"best epoch {run.get('best_epoch')}, best val_loss {run.get('best_val_loss')}", True))
    out.append((f'03 {t}: run record matches the checkpoint and the weights file (sha256)',
                run.get('checkpoint_sha256') == sha256(files['ckpt'])
                and run.get('weights_sha256') == sha256(files['weights']),
                f"checkpoint {str(run.get('checkpoint_sha256'))[:12]}, weights {str(run.get('weights_sha256'))[:12]}",
                True))
    phases = res.get('images_by_phase') or run.get('images_by_phase') or {}
    out.append((f'03 {t}: trained on enrollment images only, tested on verification images only',
                set(phases.get('train', {})) == set(phases.get('validation', {})) == {'enrollment'}
                and set(phases.get('test', {})) == {'verification'}, str(phases), True))
    acc = res.get('test_acc_margin_free')
    out.append((f'03 {t}: test accuracy (margin-free) in results_{t}.json', isinstance(acc, float) and 0 <= acc <= 1,
                f'{acc}', True))
    if max_epochs > SMOKE_EPOCHS:
        out.append((f'03 {t}: stopped early (before the {max_epochs}-epoch cap)', run.get('stopped_early') is True,
                    f"{len(epochs)} epochs, best {run.get('best_epoch')}"
                    + ('' if run.get('stopped_early') else ': val_loss may still have been falling'), False))
    return out

def checks_seeds(models, tags):
    # every seed trained on the same classes in the same order
    idx = [load(f'{models}/class_index_{t}.json') for t in tags if os.path.exists(f'{models}/class_index_{t}.json')]
    sha = {load(f'{models}/run_{t}.json').get('class_list_sha256') for t in tags
           if os.path.exists(f'{models}/run_{t}.json')}
    return [(f'03 {", ".join(tags)}: the same class index and class-list sha256',
             len(idx) == len(tags) and all(i == idx[0] for i in idx) and len(sha) == 1, f'{len(sha)} distinct sha256',
             True)]

def checks_resume(rows_at_stop):
    t = f'{SMOKE}_resume'
    if not os.path.exists(f'{SMOKE_MODELS}/run_{t}.json'):
        return [('03 --resume test: run record', False, 'missing', True)]
    run, log = load(f'{SMOKE_MODELS}/run_{t}.json'), pd.read_csv(f'{SMOKE_MODELS}/log_{t}.csv')
    stopped_mid = rows_at_stop is not None and 1 <= rows_at_stop < RESUME_TEST_EPOCHS
    return [('03 --resume test: first run stopped after a finished epoch, before the last', stopped_mid,
             f'{rows_at_stop} epochs logged when it was stopped', True),
            ('03 --resume test: continued (resumed recorded), every epoch logged exactly once',
             run.get('resumed') is True and log.epoch.tolist() == list(range(RESUME_TEST_EPOCHS)),
             f"resumed {run.get('resumed')}, log epochs {log.epoch.tolist()}", True),
            ('03 --resume test: backup removed after the finished run', not os.path.exists(f'{SMOKE_MODELS}/backup_{t}'),
             '', False)]

def checks_04(t, models, manifest):
    path = f'{EVAL}/summary_{t}.json'
    if not os.path.exists(path):
        return [(f'04 {t}: summary', False, f'{path} missing', True)]
    s, res = load(path), load(f'{models}/results_{t}.json')
    n_test = int((pd.read_csv(manifest).split == 'test').sum())
    pred = pd.read_csv(f'{EVAL}/predictions_{t}.csv')
    acc03 = res.get('test_acc_margin_free', np.nan)
    return [(f'04 {t}: every test image evaluated', s['n_images'] == n_test, f"{s['n_images']} of {n_test}", True),
            (f'04 {t}: every image found in the manifest, folder subject = manifest subject',
             bool(pred.device.notna().all() and (pred.subject == pred.true_subject).all()), '', True),
            (f'04 {t}: accuracy agrees with 03 (within 0.002)', abs(s['accuracy_margin_free'] - acc03) < 0.002,
             f"04 {s['accuracy_margin_free']:.4f}, 03 {acc03:.4f}", False)]

def checks_15(t, models, n_subjects=None):
    mpath, spath = f'{VERIF}/verification_metrics_{t}.csv', f'{VERIF}/verification_summary_{t}.json'
    if not (os.path.exists(mpath) and os.path.exists(spath)):
        return [(f'15 {t}: metrics and summary files', False, f'{mpath} or {spath} missing', True)]
    m, s = pd.read_csv(mpath), load(spath)
    overall = m[m.subset == 'all'].set_index('mode')
    tar = ['tar_at_far_1e-2', 'tar_at_far_1e-3']
    phases = s.get('training_images_by_phase') or {}
    out = [(f'15 {t}: P2 and P3 results', {'p2', 'p3'} <= set(overall.index), ', '.join(overall.index), True),
           (f'15 {t}: EER between 0 and 0.5, TAR at FAR 1e-2 and 1e-3 present',
            bool(overall.eer.between(0, 0.5).all() and overall[tar].notna().all().all()),
            ', '.join(f'{k} EER {r.eer:.4f} TAR@1e-3 {r["tar_at_far_1e-3"]:.4f}' for k, r in overall.iterrows()), True),
           (f'15 {t}: training subjects from the class index; trained without verification images',
            s.get('training_subjects') == f'{models}/class_index_{t}.json' and not phases.get('verification')
            and bool(phases.get('enrollment')), f"{s.get('training_subjects')}, {phases}", True),
           (f'15 {t}: commit and clean checkout', same_commit(s.get('git_commit')), str(s.get('git_commit')), True)]
    if n_subjects is not None and 'p2' in overall.index:
        out.append((f'15 {t}: P2 on all {n_subjects} training subjects with test images',
                    int(overall.loc['p2', 'n_subjects']) == n_subjects, f"{overall.loc['p2', 'n_subjects']}", False))
    return out

def run_checks(*specs):
    # specs: (check function, its arguments...); a crash counts as a failed hard check
    out = []
    for fn, *args in specs:
        try:
            out += fn(*args)
        except Exception as e:
            out.append((f'{fn.__name__} crashed', False, f'{type(e).__name__}: {e}', True))
    return out

def report(checks):
    # print PASS / WARN / FAIL per check; return the failed hard checks
    for name, ok, detail, hard in checks:
        print(f"{'PASS' if ok else ('FAIL' if hard else 'WARN')}  {name}" + (f'  [{detail}]' if detail else ''))
    return [c for c in checks if c[3] and not c[1]]

## 6-8. Pre-flight checks, data and resume

Cell 6 checks everything that can be checked before the GPUs start, including that every script and flag used here is on `BRANCH` and that `configs/cross_session.yml` holds the recipe. Cell 7 builds the manifest (about 15 min; it walks the 134,258 images), the cross-session split with its tree and split lists, the open-set list for P3 and the 20-class smoke tree, then checks the split. Cell 8 does something only when `PREVIOUS_OUTPUT` is set.

In [ ]:
# 6. Pre-flight: GPUs, the merged code, its flags and the recipe, packages, data, resume input, disk
import yaml
problems = []
n_gpu = len(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout.strip().splitlines())
print('GPUs visible:', n_gpu)
if n_gpu < len(SEEDS):
    problems.append(f'{n_gpu} GPU(s) visible for {len(SEEDS)} seeds: set the accelerator to GPU T4 x2')

# the training-recipe PR: every script and flag used here must be on BRANCH
scripts = sorted({t for cmd in CMD.values() for t in cmd.split() if t.startswith('scripts/')})
missing = [p for p in scripts + [CFG] if not os.path.exists(f'{REPO}/{p}')]
if missing:
    problems.append(f'{BRANCH} lacks {missing}: merge the training-recipe PR first (or set BRANCH to a branch that has it)')
for path in scripts:
    if os.path.exists(f'{REPO}/{path}'):
        text = open(f'{REPO}/{path}').read()
        used = {t for cmd in CMD.values() if path in cmd.split()
                for t in cmd.split()[cmd.split().index(path) + 1:] if t.startswith('--')}
        lost = sorted(f for f in used if f'"{f}"' not in text and f"'{f}'" not in text)
        if lost:
            problems.append(f'{path} has no {lost}: merge the training-recipe PR first')
MAX_EPOCHS = None
if os.path.exists(f'{REPO}/{CFG}'):
    recipe = yaml.safe_load(open(f'{REPO}/{CFG}'))['training']
    MAX_EPOCHS = int(recipe['epochs'])
    print(f'{CFG}:', recipe)
    want = {'epochs': 30, 'early_stop_patience': 5, 'early_stop_restore_best': True, 'reduce_lr_patience': 2,
            'reduce_lr_factor': 0.2, 'min_lr': 1e-6}
    differ = {k: recipe.get(k) for k, v in want.items() if recipe.get(k) != v}
    if differ:
        print('NOTE: the recipe differs from the one this notebook was written for:', differ)
    if not recipe['min_lr'] < recipe['learning_rate']:
        problems.append(f'{CFG}: min_lr must be below learning_rate, or the learning-rate schedule does nothing')

# packages, checked in their own process
code, tfv = py("import os; os.environ['TF_USE_LEGACY_KERAS'] = '1'; import importlib.metadata as m, tensorflow as tf; "
               "print(tf.__version__, m.version('tf-keras'), len(tf.config.list_physical_devices('GPU')))")
print('tensorflow, tf-keras, GPUs seen by TensorFlow:', tfv)
parts = tfv.split()
if code or len(parts) != 3 or not parts[0].startswith('2.20.') or parts[1] != '2.20.1':
    problems.append(f'TensorFlow 2.20 with tf-keras 2.20.1 expected (the versions of the baseline run): {tfv}')

# data, the earlier version's output (resume), disk space
VISOB = next((os.path.join(r, 'VISOB_Augmented_Recent') for r, dirs, _ in os.walk('/kaggle/input')
              if 'VISOB_Augmented_Recent' in dirs), None)
print('VISOB data:', VISOB)
if VISOB is None:
    problems.append('VISOB_Augmented_Recent not found under /kaggle/input: add mubashirulhassan00/visob-augmented-recent')
PREV = None
if PREVIOUS_OUTPUT:
    found = sorted(glob.glob(f"{PREVIOUS_OUTPUT.rstrip('/')}/**/trained_models", recursive=True))
    PREV = os.path.dirname(found[0]) if len(found) == 1 else None
    print('earlier output:', PREV, '| trained_models folders found:', found)
    if PREV is None or not os.path.isdir(f'{PREV}/splits'):
        problems.append(f'PREVIOUS_OUTPUT = {PREVIOUS_OUTPUT}: expected exactly one trained_models folder with a splits '
                        f'folder next to it (attach the stopped version\'s output)')
free = shutil.disk_usage(W).free / 2**30
print(f'free space in {W}: {free:.1f} GB')
if free < 12:     # 3 smoke models of about 2.3 GB, then per seed a 1.7 GB checkpoint, 0.6 GB weights, a ~1.8 GB backup
    problems.append(f'only {free:.1f} GB free in {W} (12 GB needed)')
print('\n'.join(['PROBLEMS:'] + problems) if problems else f'pre-flight ok ({hours():.2f} h into the session)')
assert not problems, 'fix the problems above, then run this cell again'

In [ ]:
# 7. Manifest, the cross-session split (tree, manifest, split lists), the open-set list and the smoke tree; checks
t0 = time.time()
FG['01_manifest'] = sh(CMD['01_manifest'])            # about 15 min
FG['02_cross_session'] = sh(CMD['02_cross_session'])  # tree /tmp/ocular_cross_session, manifest_split_cross_session.csv
FG['02_open_set'] = sh(CMD['02_open_set'])            # splits/open_set_test.csv: subjects outside the 300 classes (P3)
FG['02_small'] = sh(CMD['02_small'])                  # the first SMOKE_CLASSES classes, for the smoke runs
print(f'data steps: {(time.time() - t0) / 60:.1f} min')

raw = pd.read_csv(f'{W}/manifest_raw.csv')
print('manifest rows:', len(raw))
assert len(raw) == 134258, 'the VISOB copy is not the version the baseline was trained on (134,258 images): stop'
for name, manifest in (('cross-session', CS_MANIFEST), ('smoke tree', SMALL_MANIFEST)):
    print(f'\n{name}: images per split and phase\n{split_counts(manifest).to_string()}')
COUNTS = {name: pd.read_csv(manifest).split.value_counts().to_dict()
          for name, manifest in (('full', CS_MANIFEST), ('small', SMALL_MANIFEST))}
print(f"open-set list: {len(pd.read_csv(OPEN_SET))} images of subjects outside the 300 classes")
failed = report(run_checks((checks_split, CS_MANIFEST, N_CLASSES), (checks_split_csvs,),
                           (checks_split, SMALL_MANIFEST, SMOKE_CLASSES)))
assert not failed, 'the split is not what the protocol promises: stop'

In [ ]:
# 8. Resume (only with PREVIOUS_OUTPUT): copy the earlier models, backups and results; the split must be identical
def training_done(t):
    # when training ends, 03 writes run_<tag>.json with finished = true and removes the backup (run again, it
    # would train nothing)
    path = f'{MODELS}/run_{t}.json'
    return os.path.exists(path) and load(path).get('finished') is True and not os.path.isdir(f'{MODELS}/backup_{t}')

def epochs_logged(t):
    return csv_rows(f'{MODELS}/log_{t}.csv')

if PREV:
    for split in ('train', 'validation', 'test'):
        old, new = pd.read_csv(f'{PREV}/splits/cross_session_{split}.csv'), pd.read_csv(f'{SPLITS}/cross_session_{split}.csv')
        assert old.equals(new), f'the {split} split differs from the earlier session: resuming would train on other images'
    for folder in ('trained_models', 'eval_clean', 'verification'):
        if os.path.isdir(f'{PREV}/{folder}'):
            # copyfile: new, writable files (inputs are read-only), so 03 can update and then remove the backups
            shutil.copytree(f'{PREV}/{folder}', f'{W}/{folder}', copy_function=shutil.copyfile, dirs_exist_ok=True)
    subprocess.run(['chmod', '-R', 'u+w', MODELS, EVAL, VERIF], capture_output=True)
    if os.path.isdir(f'{PREV}/logs'):
        shutil.copytree(f'{PREV}/logs', f'{LOGS}/earlier_session', copy_function=shutil.copyfile, dirs_exist_ok=True)
    for s in SEEDS:
        t = tag(s)
        state = ('finished' if training_done(t) else
                 f'backup after {epochs_logged(t)} epochs' if os.path.isdir(f'{MODELS}/backup_{t}') else 'not started')
        print(f'{t}: training {state} | 04 done: {os.path.exists(f"{EVAL}/summary_{t}.json")} | '
              f'15 done: {os.path.exists(f"{VERIF}/verification_summary_{t}.json")}')
else:
    print('PREVIOUS_OUTPUT is None: every seed trains from the start')

## 9-11. Smoke runs, checks and the time estimate

Cell 9 trains both seeds at the same time on the 20-class tree for 2 epochs with the full recipe flags, so the epoch time is measured with both GPUs and the shared CPU busy. Cell 10 tests `--resume` on GPU 0 (a 3-epoch training is stopped as soon as the backup of its first epoch is complete, as Kaggle's time limit would stop it, and the same command is run again), while GPU 1 runs 04 and 15 on the seed-1 smoke model. Cell 11 checks every output file, prints the estimate for 30 epochs and refuses the full runs if both seeds would not finish before `HARD_STOP_H`. The estimate scales the second (steady) smoke epoch by the ratio of training images, except the seconds it spends writing the checkpoint and the backup (measured in cell 10; they take as long for the full model); it is an upper bound because it assumes all 30 epochs.

In [ ]:
# 9. Smoke A: every seed's smoke training at the same time, one per GPU
if os.path.isdir(SMOKE_MODELS):
    shutil.rmtree(SMOKE_MODELS)       # a leftover backup would make the smoke trainings resume instead of start
SA = wait([launch(f'GPU {i % n_gpu}', [f'03_{tag(s, SMOKE)}'], gpu=i % n_gpu) for i, s in enumerate(SEEDS)])
for s in SEEDS:
    print(f'{tag(s, SMOKE)}: epoch seconds {epoch_seconds(f"03_{tag(s, SMOKE)}")} | step {seconds(f"03_{tag(s, SMOKE)}")} s')

In [ ]:
# 10. Smoke B: the --resume test on GPU 0 (stopped after its first epoch, then continued); 04 and 15 smoke on GPU 1
for path in glob.glob(f'{SMOKE_MODELS}/*{SMOKE}_resume*'):   # a rerun of this cell starts the test afresh (a
    shutil.rmtree(path) if os.path.isdir(path) else os.remove(path)   # finished test would train nothing)
gpu1 = launch(f'GPU {1 % n_gpu}', ['04_smoke', '15_smoke'], gpu=1 % n_gpu)
first = launch('GPU 0', ['03_resume_a'], gpu=0)
rt, backup, rows_at_stop, WRITE_S = f'{SMOKE}_resume', f'{SMOKE_MODELS}/backup_{SMOKE}_resume', None, None
while first['proc'].poll() is None and hours() < HARD_STOP_H:
    time.sleep(5)
    # stop once an epoch is logged and its backup is complete (BackupAndRestore is the last callback of an epoch,
    # so the checkpoint, the log row and the callback state of that epoch are written by then)
    if csv_rows(f'{SMOKE_MODELS}/log_{rt}.csv') >= 1 and backup_written(backup):
        rows_at_stop = csv_rows(f'{SMOKE_MODELS}/log_{rt}.csv')
        # seconds an epoch spends writing: the backup (from the log row to the backup's state file) and, about as
        # long, the checkpoint just before the log row (Keras counts both in the epoch time it prints)
        backup_s = max(mtime(p) for p in glob.glob(f'{backup}/**/checkpoint', recursive=True)) \
            - mtime(f'{SMOKE_MODELS}/log_{rt}.csv')
        WRITE_S = 2 * backup_s if backup_s >= 0 else None     # < 0: the log has a newer row than the backup
        print(f'stopping the first run after {rows_at_stop} logged epoch(s), as the 12-hour limit would '
              f'(checkpoint + backup writes: about {WRITE_S} s per epoch)')
        stop([first])
        break
if rows_at_stop is None:
    print('the first run ended before its first backup was complete:', status('03_resume_a'))
    print('\n'.join(tail(f'{LOGS}/03_resume_a.log', 40)))
SB = wait([launch('GPU 0', ['03_resume_b'], gpu=0)], also=[gpu1]) if rows_at_stop is not None else {}
SB.update(wait([gpu1]))

In [ ]:
# 11. Smoke checks, the time estimate for the full runs and the 12-hour check
smoke_tags = [tag(s, SMOKE) for s in SEEDS]
smoke_failed = report(run_checks(*[(checks_03, t, SMOKE_MODELS, SMOKE_CLASSES, SMOKE_EPOCHS, s)
                                   for s, t in zip(SEEDS, smoke_tags)],
                                 (checks_seeds, SMOKE_MODELS, smoke_tags), (checks_resume, rows_at_stop),
                                 (checks_04, s1, SMOKE_MODELS, SMALL_MANIFEST), (checks_15, s1, SMOKE_MODELS)))
assert all(SA[f'03_{t}'] == 0 for t in smoke_tags), '03 failed in the smoke run: fix it before the full runs'
assert not [c for c in smoke_failed if c[0].startswith('03')], 'fix the 03 problems above first'
assert SB.get('03_resume_b') == 0, '--resume did not work in the smoke test: a stopped run could not be continued'
assert SB.get('04_smoke') == 0 and SB.get('15_smoke') == 0, '04 or 15 failed in the smoke run: see their logs above'
assert not smoke_failed, 'fix the failed checks above before the full runs'

# one steady epoch: the last smoke epoch of the slowest seed (all seeds ran at the same time), scaled by images
epochs = {t: epoch_seconds(f'03_{t}') for t in smoke_tags}
assert all(epochs.values()), f'no epoch times found in the 03 smoke logs (03 must run with --verbose 2): {epochs}'
steady = max(e[-1] for e in epochs.values())
warm = max(max(e[0] - e[-1], 0) for e in epochs.values())
full, small = COUNTS['full'], COUNTS['small']
ratio = max(full['train'] / small['train'], full.get('validation', 0) / max(small.get('validation', 1), 1))
# the checkpoint and backup writes (measured in cell 10) take as long for the full model as for the smoke model:
# they are not scaled; without a measurement the whole epoch is scaled (an overestimate)
fixed = min(WRITE_S, steady) if WRITE_S is not None else 0
epoch_full = (steady - fixed) * ratio + fixed
# 03's own overhead (start, model, test evaluation of the small tree) plus its test evaluation of the extra images
# (two predictions per image), at 04's rate per image; 04 and 15 scaled by their images
v = load(f'{VERIF}/verification_summary_{s1}.json')
load_s = v['timing']['load_model_s']
per_img_04 = max(seconds('04_smoke') - load_s, 1) / small['test']
extra_test = full['test'] - small['test']
over_03 = max(max(seconds(f'03_{t}') - sum(e) for t, e in epochs.items()), 0) + 2 * per_img_04 * extra_test
n15 = int((raw.subject.between(1141, 3279) | (raw.subject == 7000)).sum()) + len(pd.read_csv(OPEN_SET))
EST = {'epoch_min': epoch_full / 60, 'write_s_per_epoch': WRITE_S,
       '04_h': (seconds('04_smoke') + per_img_04 * extra_test) / 3600,
       '15_h': (load_s + v['timing']['embedding_s'] / v['n_images_embedded'] * n15 + 600) / 3600}
plan = {}
for s in SEEDS:
    t = tag(s)
    left = 0 if training_done(t) else MAX_EPOCHS - epochs_logged(t)
    plan[t] = ((left * epoch_full + warm + over_03) / 3600 if left else 0)
    plan[t] += (0 if os.path.exists(f'{EVAL}/summary_{t}.json') else EST['04_h'])
    plan[t] += (0 if os.path.exists(f'{VERIF}/verification_summary_{t}.json') else EST['15_h'])
EST['per_gpu_h'] = max(plan.values())
finish = hours() + EST['per_gpu_h'] + 0.3      # + about 20 min for the checkpoint hashes, results.zip and the checklist
print(f"\nsmoke epochs (s): {epochs} | steady epoch {steady} s, of which {fixed:.0f} s writing the checkpoint and "
      f"the backup (not scaled{'' if WRITE_S is not None else ': not measured, so the estimate is high'}); "
      f"the rest x {ratio:.1f} (images) = {EST['epoch_min']:.1f} min per full epoch (random split on 25/27 Oct: "
      f"about 17 min)")
print(f"03 for {MAX_EPOCHS} epochs (the cap): about {(MAX_EPOCHS * epoch_full + warm + over_03) / 3600:.1f} h; "
      f"if early stopping ends it at epoch 15: about {(15 * epoch_full + warm + over_03) / 3600:.1f} h")
print(f"04: about {EST['04_h'] * 60:.0f} min | 15 (P2 + P3, {n15} images): about {EST['15_h'] * 60:.0f} min")
print(f"per GPU (one seed each, in parallel), what is left: {', '.join(f'{t} {h:.1f} h' for t, h in plan.items())}; "
      f"the session would end at about {finish:.1f} h of the 12 (HARD_STOP_H = {HARD_STOP_H})")
if RUN_FULL:
    # room for the full runs: the smoke checkpoints (about 2.3 GB each) are no longer needed; their records stay
    for path in glob.glob(f'{SMOKE_MODELS}/*.keras') + glob.glob(f'{SMOKE_MODELS}/*.h5'):
        os.remove(path)
if RUN_FULL and finish >= HARD_STOP_H:
    assert START_IF_TOO_LONG, ('the full runs might not finish in this session even with early stopping: set '
                               'START_IF_TOO_LONG = True to train until HARD_STOP_H and resume in a new session '
                               '(see Resume), or train fewer seeds per session')
    print('WARNING: the estimate ends after HARD_STOP_H; START_IF_TOO_LONG is True, so the runs start and continue '
          'in the next session')

## 12. The full runs (`RUN_FULL = True`)

One chain per GPU: 03 trains seed 1 (GPU 0) and seed 2 (GPU 1) on the cross-session tree with `configs/cross_session.yml`, `--resume` (a backup after every epoch), `--reshuffle` (new batches every epoch) and `--save-weights` (a weights-only copy of the best model), into `trained_models/`. Then 04 evaluates the best checkpoint on the cross-session test split (every verification image) with the training class order (`--class-index`), and 15 runs P2 (templates from enrollment images vs verification probes, the 300 training subjects) and P3 (the same for the subjects never trained on). Steps that an earlier session finished are skipped.

In [ ]:
# 12. Full runs (RUN_FULL): per seed and GPU, 03 -> 04 -> 15 P2 + P3; steps finished in an earlier session are skipped
def mark_done(name, why):
    # a step an earlier session finished: logged as succeeded, for the steps that need it and for the checklist
    with open(f'{LOGS}/{name}.log', 'w') as f:
        f.write(why + '\n')
    with open(f'{LOGS}/{name}.exit', 'w') as f:
        f.write('0\n')

C = {}
if RUN_FULL:
    jobs = []
    for i, s in enumerate(SEEDS):
        t = tag(s)
        trained = training_done(t)
        steps = []
        for name, out in ((f'03_{t}', f'{MODELS}/run_{t}.json'), (f'04_{t}', f'{EVAL}/summary_{t}.json'),
                          (f'15_{t}', f'{VERIF}/verification_summary_{t}.json')):
            if trained and os.path.exists(out):    # 04 and 15 outputs count only for a finished training
                mark_done(name, f'done in an earlier session ({out})')
            else:
                steps.append(name)
        if steps:
            jobs.append(launch(f'GPU {i % n_gpu}', steps, gpu=i % n_gpu))
    C = wait(jobs, every=300) if jobs else {}
else:
    print('RUN_FULL is False: full runs skipped. Set RUN_FULL = True (cell 2) and use Save Version > Save & Run All.')

## 13-15. Results table, results.zip and the final checklist

Cell 13 collects one row per seed (epochs, best epoch, test accuracy, P2 and P3 EER, TAR at FAR 1e-2 and 1e-3, rank-1) and the mean and standard deviation over the seeds. `results.zip` (cell 14) is written before the checklist, so a failed check never loses the outputs. It holds the records, logs, split lists and results but **no checkpoints**: they stay in the Output panel (listed with their sha256). The checklist (cell 15) asserts the expected files and the checks above.

In [ ]:
# 13. One row per seed and the mean / standard deviation over the seeds -> summary_cross_session.csv
rows = []
for s in SEEDS:
    t = tag(s)
    row = {'seed': s, 'tag': t}
    if os.path.exists(f'{MODELS}/run_{t}.json'):
        run = load(f'{MODELS}/run_{t}.json')
        row.update({k: run.get(k) for k in ('epochs_run', 'best_epoch', 'best_val_loss', 'stopped_early',
                                            'minutes_all_sessions', 'resumed')})
    if os.path.exists(f'{MODELS}/results_{t}.json'):
        row['test_acc_margin_free_03'] = load(f'{MODELS}/results_{t}.json').get('test_acc_margin_free')
    if os.path.exists(f'{EVAL}/summary_{t}.json'):
        row['test_acc_margin_free_04'] = load(f'{EVAL}/summary_{t}.json')['accuracy_margin_free']
    if os.path.exists(f'{VERIF}/verification_summary_{t}.json'):
        for mode, r in load(f'{VERIF}/verification_summary_{t}.json')['per_mode'].items():
            for k in ('eer', 'tar_at_far_1e-2', 'tar_at_far_1e-3', 'rank1'):
                row[f'{mode}_{k}'] = r.get(k)
    rows.append(row)
table = pd.DataFrame(rows)
numeric = table.drop(columns=['seed', 'tag', 'stopped_early', 'resumed'], errors='ignore')
numeric = numeric.apply(pd.to_numeric, errors='coerce')
if len(table) > 1:
    table = pd.concat([table, pd.DataFrame([{'tag': 'mean', **numeric.mean().to_dict()},
                                            {'tag': 'std', **numeric.std().to_dict()}])], ignore_index=True)
table['seed'] = table['seed'].astype('Int64')
table.to_csv(f'{W}/summary_cross_session.csv', index=False)
with pd.option_context('display.width', 250, 'display.max_columns', 40):
    print(table.round(4).to_string(index=False))

In [ ]:
# 14. results.zip without checkpoints (download it from the Output panel; never commit or upload it)
import zipfile
info = {'branch': BRANCH, 'commit': COMMIT, 'run_full': RUN_FULL, 'tag': TAG, 'smoke_tag': SMOKE,
        'settings': {'seeds': SEEDS, 'max_frame': MAX_FRAME, 'cfg': CFG, 'max_epochs': MAX_EPOCHS, 'smoke_classes': SMOKE_CLASSES,
                     'smoke_epochs': SMOKE_EPOCHS, 'n_boot': N_BOOT, 'previous_output': PREVIOUS_OUTPUT,
                     'start_if_too_long': START_IF_TOO_LONG},
        'commands': CMD, 'steps': {**{n: status(n) for n in CMD if n not in FG}, **FG}, 'counts': COUNTS,
        'estimates_h': globals().get('EST', {}), 'session_hours': round(hours(), 2)}
with open(f'{W}/run_info.json', 'w') as f:
    json.dump(info, f, indent=2)
big = ('.keras', '.h5')
left_out = []
with zipfile.ZipFile(f'{W}/results.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for name in ('run_info.json', 'summary_cross_session.csv'):
        z.write(f'{W}/{name}', name)
    for folder in ('trained_models', 'smoke_models', 'eval_clean', 'verification', 'splits', 'logs'):
        for root, dirs, files in os.walk(f'{W}/{folder}'):
            dirs[:] = [d for d in dirs if not d.startswith('backup_')]     # training backups stay out
            for name in sorted(files):
                path = os.path.join(root, name)
                if name.endswith(big) or os.path.getsize(path) > ZIP_MAX_MB * 2**20:
                    left_out.append(os.path.relpath(path, W))
                else:
                    z.write(path, os.path.relpath(path, W))
print(f"results.zip: {os.path.getsize(f'{W}/results.zip') / 2**20:.1f} MB")
print('\nleft out of results.zip (download them separately from the Output panel):')
for rel in left_out:
    print(f'  {rel}  ({os.path.getsize(f"{W}/{rel}") / 2**30:.2f} GB)')
backups = sorted(glob.glob(f'{MODELS}/backup_*'))
if backups:
    print('\ntraining backups (a stopped run; needed to resume):', [os.path.basename(b) for b in backups])
print('\nTo keep the checkpoints for later notebooks: after this version is saved, either attach its output as an input '
      '(Add Input > Your Work > this notebook) or create a dataset from the version\'s output and set it to Private. '
      'Never make the notebook, its output or that dataset public, and share it only with the team\'s Kaggle accounts.')

In [ ]:
# 15. Final checklist: expected files and sanity checks; the run fails here if a hard check fails
if RUN_FULL:
    tags = [tag(s) for s in SEEDS]
    test_classes = pd.read_csv(CS_MANIFEST).query("split == 'test'").subject.nunique()
    checks = run_checks((checks_split, CS_MANIFEST, N_CLASSES), (checks_split_csvs,),
                        *[(checks_03, t, MODELS, N_CLASSES, MAX_EPOCHS, s) for s, t in zip(SEEDS, tags)],
                        (checks_seeds, MODELS, tags),
                        *[(checks_04, t, MODELS, CS_MANIFEST) for t in tags],
                        *[(checks_15, t, MODELS, test_classes) for t in tags])
    expected = ['01_manifest', '02_cross_session', '02_open_set', '02_small'] + [f'03_{t}' for t in smoke_tags] + \
        ['03_resume_b', '04_smoke', '15_smoke'] + [f'{step}_{t}' for t in tags for step in ('03', '04', '15')]
else:
    checks = run_checks((checks_split, CS_MANIFEST, N_CLASSES), (checks_split_csvs,),
                        *[(checks_03, t, SMOKE_MODELS, SMOKE_CLASSES, SMOKE_EPOCHS, s)
                          for s, t in zip(SEEDS, smoke_tags)],
                        (checks_seeds, SMOKE_MODELS, smoke_tags), (checks_resume, rows_at_stop),
                        (checks_04, s1, SMOKE_MODELS, SMALL_MANIFEST), (checks_15, s1, SMOKE_MODELS))
    expected = ['01_manifest', '02_cross_session', '02_open_set', '02_small'] + [f'03_{t}' for t in smoke_tags] + \
        ['03_resume_b', '04_smoke', '15_smoke']
steps = {n: FG.get(n, status(n)) for n in expected}
bad = {n: s for n, s in steps.items() if s != 0}
checks.append(('every step of the session succeeded', not bad, ', '.join(f'{n}: {s}' for n, s in bad.items()), True))
checks.append(('results.zip written', os.path.exists(f'{W}/results.zip'), '', True))
failed = report(checks)
warned = [c for c in checks if not c[1] and not c[3]]
record = {'run_full': RUN_FULL, 'commit': COMMIT, 'session_hours': round(hours(), 2), 'steps': steps,
          'checks': [{'check': n, 'passed': bool(ok), 'detail': d, 'hard': h} for n, ok, d, h in checks]}
name = f"checklist_{TAG if RUN_FULL else SMOKE}.json"
with open(f'{W}/{name}', 'w') as f:
    json.dump(record, f, indent=2)
with zipfile.ZipFile(f'{W}/results.zip', 'a') as z:
    z.write(f'{W}/{name}', name)
print(f'\n{len(checks) - len(failed) - len(warned)} passed, {len(warned)} warnings, {len(failed)} failed | '
      f'session {hours():.1f} h')
if RUN_FULL and glob.glob(f'{MODELS}/backup_*'):
    print('A training was stopped before it finished: resume it in a new session (set PREVIOUS_OUTPUT, see the top).')
assert not failed, 'some checks failed (listed above); the outputs and results.zip are saved'

## After the run

1. Download `results.zip` and unzip it into a new folder under `D:\TempCLG\results` on the laptop. Never commit it or put it on the web; back it up in the team's private storage.
2. Keep the checkpoints (`trained_models/Xception_ArcFace_cs_s1.keras`, `..._cs_s2.keras` and the smaller `.weights.h5` copies) in Kaggle: attach this version's output to later notebooks, or make a **private** dataset from it and share it with the team's accounts only. Their sha256 is in `run_cs_s<seed>.json`.
3. Report the cross-session numbers as mean and spread over the two seeds (`summary_cross_session.csv`): test accuracy (margin-free) on the verification images, and P2 / P3 EER and TAR at FAR 1e-2 / 1e-3 with their bootstrap CIs (`verification/verification_metrics_cs_s<seed>.csv`). Do not compare them with the 97.96 % of the random split: that split puts near-duplicate frames on both sides.
4. If a run stopped at the 30-epoch cap (a warning in the checklist), val_loss may still have been falling: say so, or train longer in a new session.
5. If Kaggle stopped the session, resume as described at the top (`PREVIOUS_OUTPUT`).